In [1]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObserveType, RewardOptions

num_envs, total_episodes = 16, 100000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObserveType.VEC_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
            reward_options=RewardOptions(
                eats_apple=24.0,
                penalty_step=-0.01,
                penalty_loop=-0.1,
                death_wall=-20.0,
                death_self=-20.0,
                shaping_closer=0.1,
                shaping_further=-0.1,
                complete=100.0,
            ),
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01) ** (1 / (total_episodes * 0.6))

training_logs = []
episode_rewards = np.zeros(num_envs)
completed = 0
states, infos = env.reset()
best_reward = -np.inf

In [3]:
from agents.dqn import DQNAgent

agent_name = "dqn_snake_2"
agent = DQNAgent(
    lr=0.05,
    hidden_size=256,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

In [4]:
with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        actions = [agent.act(s) for s in states]
        next_states, rewards, terminated, truncated, next_infos = env.step(actions)
        for i in range(len(actions)):
            s = states[i]
            a = actions[i]
            r = float(rewards[i])
            ns = next_states[i]
            done_i = bool(terminated[i])
            trunc_i = bool(truncated[i])

            agent.train_short_memory(s, a, r, ns, done_i)
            agent.remember(s, a, r, ns, done_i)
            episode_rewards[i] += r

            if done_i or trunc_i:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train_long_memory()
                    agent.train()
                    training_logs.append(
                        {
                            "episode": completed,
                            "reward": float(episode_rewards[i]),
                            "epsilon": agent.epsilon,
                        }
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean([log["reward"] for log in training_logs[-100:]])
                        if recent_avg > best_reward:
                            best_reward = recent_avg

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else float(episode_rewards[i])
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0.0

        states = next_states

env.close()

Parallel Training:   0%|          | 1/100000 [00:00<6:21:12,  4.37it/s]/home/enderpalm/cedt/snake-rl/agents/dqn.py:100: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  state = torch.tensor(state, dtype=torch.float, device=self.device)
Parallel Training:   3%|▎         | 3412/100000 [14:58<7:03:45,  3.80it/s]


KeyboardInterrupt: 